# Chapter 4 full experiments

Upload/extract the completed version 0.2 repository first, then prepare the complete canonical datasets using README.md. No pilot stage or fabricated benchmark results are used. Preserve selection/, configs/frozen/, runs/ and experiments/ across shutdowns. This notebook does not bypass notebook quotas.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
repo = Path('/content/drl-context-poi')
if not (repo / 'src/drlpoi/upstream.py').exists():
    raise FileNotFoundError('Extract the completed version 0.2 repository to ' + str(repo))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo) + '[dev,plots,baselines]'], check=True)
subprocess.run([sys.executable, 'scripts/check_environment.py'], cwd=repo, check=True)
for model in ('sasrec', 'getnext', 'dchl', 'ipcm'):
    subprocess.run([sys.executable, 'scripts/fetch_upstream.py', model], cwd=repo, check=True)


## Full-data validation selection

Prepare data/canonical/{nyc,tky,gowalla_ca}.csv and audit them first. The cell below runs bounded selection on seed 11 and all fitting data, with no test evaluation. Run while GPU 0 is idle. Existing frozen configurations are preserved; deliberately changing the protocol requires a new experiment directory.

In [ ]:
for dataset in ('nyc','tky','gowalla_ca'):
    csv = repo / 'data/canonical' / (dataset + '.csv')
    if not csv.exists():
        raise FileNotFoundError(csv)
    frozen = repo / 'configs/frozen' / (dataset + '.yaml')
    if not frozen.exists():
        config = 'configs/gowalla.yaml' if dataset == 'gowalla_ca' else 'configs/base.yaml'
        subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'calibrate', '--csv', str(csv),
                        '--config', config, '--out', 'selection/' + dataset, '--export', str(frozen)],
                       cwd=repo, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), check=True)


## Independent training workers

Each job uses one GPU. The two-T4 setup does not pool VRAM. Monitor worker logs. Reuse the unchanged plan and completed artifacts to resume. Remove stale locks only after confirming their processes have stopped.

In [ ]:
plan = repo / 'experiments/final/plan.json'
if not plan.exists():
    subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'plan', '--suite', 'full',
                    '--settings-dir', 'configs/frozen', '--out', 'experiments/final'], cwd=repo, check=True)
workers, handles = [], []
for gpu in ['0']:
    log = open(repo / ('worker_gpu' + gpu + '.log'), 'a')
    handles.append(log)
    workers.append(subprocess.Popen([sys.executable, '-m', 'drlpoi.cli', 'worker',
                    '--plan', str(plan), '--gpu', gpu], cwd=repo, stdout=log, stderr=subprocess.STDOUT))
print('Worker PIDs:', [p.pid for p in workers])


## Reference timing

Run only after all training workers have exited successfully and all other GPU jobs are stopped. The worker enforces completion of the non-timing phase. Three timing repeats use one checkpoint; they are not training seeds.

In [ ]:
if any(p.poll() is None for p in workers):
    raise RuntimeError('Training workers are still active; wait for completion before timing.')
if any(p.returncode != 0 for p in workers):
    raise RuntimeError('Inspect failed worker logs before continuing.')
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'worker', '--plan', str(plan), '--gpu', '0', '--timing'],
               cwd=repo, check=True)


## Reports

All figures/statistics below use completed real runs. Do not fill missing entries with synthetic test outputs. Report single-seed supplementary analyses and baseline adaptations explicitly.

In [ ]:
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'report', '--root', 'runs', '--out', 'artifacts/report'], cwd=repo, check=True)
for dataset in ('nyc','tky','gowalla_ca'):
    subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'compare-family', '--family',
                    'experiments/final/family_' + dataset + '.json', '--out',
                    'artifacts/report/paired_' + dataset + '.json'], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'diagnostics', '--root', 'runs',
                '--out', 'artifacts/report/discussions.json'], cwd=repo, check=True)
